# Healthcare Patient Deterioration Analytics
**Student:** Vansh Jain \
**Technologies Used:** Azur Data Factory · Databricks PySpark · Unity Catalog · Spark MLlib \
**Notebook:** Silver Layer\
**Layers:** Bronze → Silver → Gold


### Importing Libraries

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    lit,
    count,
    trim,
    lower,
    round
)

### Storage Account Configuration

Connecting Storage account to azure DataBricks

In [0]:
storage_account = 'healthcaremedalliondev'
storage_key = '<Your Account Key>'

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
bronze_path = f'abfss://bronze@{storage_account}.dfs.core.windows.net/bronze_patient_vitals'
silver_path = f'abfss://silver@{storage_account}.dfs.core.windows.net/silver_patient_vitals'
reject_path = f'abfss://silver@{storage_account}.dfs.core.windows.net/silver_rejected_records'

### Read Data

Read data from bronze container in ADLS

In [0]:
bronze_df = spark.read.format('delta').load(bronze_path)

### Basic Data Validation
Performing Schema checks and Ensuring no data got lost

In [0]:
bronze_df.count()
bronze_df.printSchema()
display(bronze_df.limit(5))

In [0]:
expected_columns = [
    'hour_from_admission',
    'heart_rate',
    'respiratory_rate',
    'spo2_pct',
    'temperature_c',
    'systolic_bp',
    'diastolic_bp',
    'oxygen_device',
    'oxygen_flow',
    'mobility_score',
    'nurse_alert',
    'wbc_count',
    'lactate',
    'creatinine',
    'crp_level',
    'hemoglobin',
    'sepsis_risk_score',
    'age',
    'gender',
    'comorbidity_index',
    'admission_type',
    'deterioration_next_12h',
    'ingestion_timestamp',
    'source_file_name'
]

missing = set(expected_columns) - set(bronze_df.columns)

if missing:
    raise Exception(f'Missing Columns : {missing}')

### Basic Data Cleaning Operations

In [0]:
duplicate_count = (
    bronze_df
    .groupBy(bronze_df.columns)
    .count()
    .filter(col('count') > 1)
    .count()
)

print(f'Duplicate Records : {duplicate_count}')

In [0]:
from pyspark.sql.functions import sum

null_summary = bronze_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in bronze_df.columns
])

display(null_summary)

### Applying Business Logics

Rejecting Data which do not fit in actual data range

In [0]:
validated_df = bronze_df.withColumn(
    'rejection_reason',
    when((col('heart_rate') < 20) | (col('heart_rate') > 250), 'Invalid Heart Rate')
    .when((col('respiratory_rate') < 5) | (col('respiratory_rate') > 60), 'Invalid Respiratory Rate')
    .when((col('spo2_pct') < 50) | (col('spo2_pct') > 100), 'Invalid SpO2')
    .when((col('temperature_c') < 30) | (col('temperature_c') > 45), 'Invalid Temperature')
    .when((col('systolic_bp') < 50) | (col('systolic_bp') > 250), 'Invalid Systolic BP')
    .when((col('diastolic_bp') < 30) | (col('diastolic_bp') > 150), 'Invalid Diastolic BP')
    .when((col('age') < 0) | (col('age') > 120), 'Invalid Age')
)

### Generating Silver DataFrame

This Data Frame contains data which match Business logic

In [0]:
silver_df = validated_df.filter(
    col('rejection_reason').isNull()
)

### Reject DataFrame

Contains those Data that do not match Business Logic

In [0]:
reject_df = validated_df.filter(
    col('rejection_reason').isNotNull()
)

In [0]:
print(f'Valid Records : {silver_df.count()}')
print(f'Rejected Records : {reject_df.count()}')
print(f'Total Records : {validated_df.count()}')

In [0]:
reject_df.groupby('rejection_reason').count().show()

### Creating Engineered Features
Engineered Features: \
•	is_fever \
•	is_tachycardia \
•	is_hypoxic \
•	high_sepsis_risk \
•	pulse_pressure 


In [0]:
# Pulse Pressure
silver_df = silver_df.withColumn(
    "pulse_pressure",
    round(col("systolic_bp") - col("diastolic_bp"), 2)
)

In [0]:
# Is Fever
silver_df = silver_df.withColumn(
    'is_fever',
    when(col('temperature_c') > 38, 1)
    .otherwise(0)
)

In [0]:
# Is Tachycardia
silver_df = silver_df.withColumn(
    'is_tachycardia',
    when(col('heart_rate') > 100, 1)
    .otherwise(0)
)

In [0]:
# Is Hypoxia
silver_df = silver_df.withColumn(
    'is_hypoxic',
    when(col("spo2_pct") < 92, 1)
    .otherwise(0)
)

In [0]:
# High Sepsis Risk
silver_df = silver_df.withColumn(
    'high_sepsis_risk',
    when(col('sepsis_risk_score') >= 0.7, 1)
    .otherwise(0)
)

In [0]:
display(
    silver_df.select(
        'heart_rate',
        'temperature_c',
        'spo2_pct',
        'sepsis_risk_score',
        'pulse_pressure',
        'is_fever',
        'is_tachycardia',
        'is_hypoxic',
        'high_sepsis_risk'
    ).limit(20)
)

### Writing silver and reject DataFrame to ADLS

In [0]:
# Droping unwanted column generated due to typo error
silver_df = silver_df.drop("is_htpoxic")

# Write operation to ADLS
(
    silver_df.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema','true')
    .save(silver_path)
)

In [0]:
(
    reject_df.write
    .format('delta')
    .mode('overwrite')
    .save(reject_path)
)

### Write Validation
Path and Data format Check

In [0]:
silver_table = spark.read.format('delta').load(silver_path)

In [0]:
print(f'silver records : {silver_table.count()}')
silver_table.printSchema()
display(silver_table.limit(10))

In [0]:
reject_table = spark.read.format('delta').load(reject_path)
print(f'Rejected Records : {reject_table.count()}')

In [0]:
silver_df.groupBy("deterioration_next_12h").count().show()